In [127]:
!pip install scikit-learn


Defaulting to user installation because normal site-packages is not writeable


In [128]:
import pandas as pd
import numpy as np
import sklearn.feature_extraction.text as text
from sklearn.metrics.pairwise import cosine_similarity
vectorizer = text.TfidfVectorizer()

In [129]:
dataset = pd.read_csv('../data/amazon_products.csv',nrows=10000)
print('=== Dataset Loaded ===')
print(f"Shape: {dataset.shape}")
print(f"Columns: {dataset.columns.tolist()}")

=== Dataset Loaded ===
Shape: (10000, 11)
Columns: ['asin', 'title', 'imgUrl', 'productURL', 'stars', 'reviews', 'price', 'listPrice', 'category_id', 'isBestSeller', 'boughtInLastMonth']


In [130]:
# creating the product descrption from the columns
def create_product_text(row):
    """Combine product features into one text string"""
    features = []
    # Adding title
    features.append(str(row['title']))

    # Adding Category
    features.append(f"category_{row['category_id']}")

    # Add price range (helps find similarly priced products)
    if row['price'] > 0:
        if row['price'] < 50:
            features.append("budget")
        elif row['price'] < 200:
            features.append("mid_range")
        else:
            features.append("Premium")

    
    # Adding Bestseller info
    if(row['isBestSeller']):
        features.append("bestSeller")

    return " ".join(features)

# Applying to all the products
dataset['product_text'] = dataset.apply(create_product_text, axis=1)
print('Product features Created')
print("\nExample:")
print(dataset['product_text'].iloc[0])


Product features Created

Example:
Sion Softside Expandable Roller Luggage, Black, Checked-Large 29-Inch category_104 mid_range


In [131]:
# Converting text to numerical numbers
vectorizer = text.TfidfVectorizer(stop_words='english', max_features=5000)
tfidf_matrix = vectorizer.fit_transform(dataset['product_text'])
print(f"TF-IDF matrix created")
print(f"Shape: {tfidf_matrix.shape}")
print(f"Each Product is now a {tfidf_matrix.shape[1]}-dimensional vector")

TF-IDF matrix created
Shape: (10000, 5000)
Each Product is now a 5000-dimensional vector


In [132]:
# Calualating the cosine similarities between all the products
cosine_sim = cosine_similarity(tfidf_matrix,tfidf_matrix)
print(f"Similarity matrix created")
print(f"Shape: {cosine_sim.shape}")
print(f"Similarity between  prosuct 0 and 1: {cosine_sim[0][1]:.3f}")

Similarity matrix created
Shape: (10000, 10000)
Similarity between  prosuct 0 and 1: 0.154


In [133]:
# Build the Recommender
def recommend_similar_products(product_title, n_recommendations = 5):
    """
    Find products similar to the given product
    
    Parameters:
    - product_title: Name of the product
    - n_recommendations: Number of recommendations to return
    
    Returns:
    - DataFrame with recommended products
    """
    # Find the product in our dataset
    matching_products = dataset[dataset['title'].str.contains(product_title, case=False)]
    
    if len(matching_products) == 0:
        return f"Product '{product_title}' not Found"
    # Get the first matching product's index
    product_idx = matching_products.index[0]

    # Get similarity scores for this product
    similarity_scores = list(enumerate(cosine_sim[product_idx]))

    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)

    similar_indices = [i[0] for i in similarity_scores[1:n_recommendations+1]]

    recommendations = dataset.iloc[similar_indices][['title', 'price', 'stars', 'boughtInLastMonth']]
    recommendations['similarity_score'] = [similarity_scores[i+1][1] for i in range(n_recommendations)]

    return recommendations

print("Testing Content-based Recommender\n ")
print("="*60)

Testing Content-based Recommender
 


In [134]:
# Testing the model
print("Test 1: Products similar to 'Luggage'\n")
print(recommend_similar_products("Luggage", 5))
print("\n" + "="*60 + "\n")

Test 1: Products similar to 'Luggage'

                                                 title   price  stars  \
17   Ascella X Softside Expandable Luggage with Spi...  198.41    4.5   
494  28 Inch Luggage with Spinner Wheels Softside E...  149.99    0.0   
319  Cambridge Lightweight Luggage Softside Expanda...  164.99    4.3   
380  Bromley Softside Expandable Spinner Luggage, B...  172.19    4.5   
24   Anzio Softside Expandable Spinner Luggage, Tea...   98.62    4.1   

     boughtInLastMonth  similarity_score  
17                 100          0.576787  
494                  0          0.460594  
319                  0          0.445873  
380                  0          0.430355  
24                 300          0.420140  




In [135]:
print("Test 2: Products similar to 'Phone'\n")
print(recommend_similar_products("Phone", 5))
print("\n" + "="*60 + "\n")

Test 2: Products similar to 'Phone'

                                                  title   price  stars  \
893   Travel Painting Travel Luggage Cover Suitcase ...   36.49    0.0   
215   Carry On Luggage 20'' Travel Suitcase Rolling ...  169.99    4.8   
59    Large Luggage Suitcase with Wheels, Check-in 2...  129.99    4.8   
293   Luggage Suitcase with Spinner Wheels, 24'' Che...  199.99    4.4   
3097  Sports Baseball Sliding Shorts with Cup Holder...   16.99    4.3   

      boughtInLastMonth  similarity_score  
893                   0          0.296508  
215                   0          0.289847  
59                   50          0.259022  
293                   0          0.257343  
3097                200          0.248621  




In [ ]:
sample_product = dataset['title'].iloc[100]
print(f"Test 3: Products similar to '{sample_product}'\n")
print(recommend_similar_products(sample_product[:30], 5))

Test 3: Products similar to 'Momentous 25" Hardside Checked 8 Wheel Expandable Spinner, Tibet Lan'

                                                 title   price  stars  \
310  Momentous 30" Hardside Checked 8 Wheel Expanda...  149.57    4.7   
428  Jessica Hardside Expandable Luggage with Spinn...  139.99    4.3   
330  Jupiter 32" Hardside Checked 8 Wheel Expandabl...  139.99    5.0   
415  Jupiter 28" Hardside Checked 8 Wheel Expandabl...  119.99    5.0   
826  Helixian 27" Hardside Checked 8 Wheel Expandab...   79.97    3.2   

     boughtInLastMonth  similarity_score  
310                  0          0.561382  
428                  0          0.448204  
330                  0          0.393321  
415                  0          0.385886  
826                  0          0.383805  
